# Merge Radiomics Features with Clinical Data
Join the radiomics features CSV with the clinical labels from bd_estudiUPF.csv.
Output: a merged table with study_id, radiomics features, and clinical outcome columns.

In [1]:
import pandas as pd
import os

print("Imports OK")

Imports OK


In [2]:
# load radiomics features
radiomics_path = os.path.join("reports", "12_radiomics_features_k3_i1.csv")
radiomics_df = pd.read_csv(radiomics_path)
print(f"Radiomics: {radiomics_df.shape[0]} rows, {radiomics_df.shape[1]} columns")

Radiomics: 134 rows, 94 columns


In [3]:
# load clinical data
clinical_path = os.path.join("..", "data", "bd_estudiUPF.csv")
clinical_df = pd.read_csv(clinical_path)

# clean the study id column
clinical_df["id estudio"] = clinical_df["id estudio"].astype(str).str.strip()

print(f"Clinical: {clinical_df.shape[0]} rows, {clinical_df.shape[1]} columns")
print(f"Columns: {clinical_df.columns.tolist()}")

Clinical: 138 rows, 57 columns
Columns: ['id estudio', 'Unnamed: 1', 'Protocol', 'id paciente', 'sexo (1:female, 0: male)', 'DONANTE sexo  (1:female, 0: male, 2: no consta)', 'DONANTE edad', 'tiempo isquemia (h) nc= no consta', 'fecha TXP', 'Fecha ECO', 'Años preTPX', 'Meses pTXP', 'Días pTXP', 'motivo (1: 1 semana, 2: 1 mes, 3: 1 año, 4: sospecha, 5: seguimiento)=', 'PRIMERAS ECOGRAFIAS (5 primeras ecografias de los radiologis)', 'adjunto', 'ARFI mediana', 'ARFI media', 'ARFI DE', 'ARFI RIQ', 'Unnamed: 20', 'PE', 'WiAUC', 'RT', 'mTTI', 'TTP', 'WiR', 'WiPi', 'WoAUC', 'WiWoAUC', 'FT', 'WoR', 'QOF', 'Area', 'Unnamed: 34', 'BIOPSIA', 'Rechazo confirmadopor biopsia', 'ADEQUADA (2: limite)', 'RECHAZO biopsia (1: cel ag leve, 2: cel ag mod, 3: cel ag sev, 4: humoral, 5:  indet)', 'RECHAZO CLÍNICO', 'fibrosis intersticial (Masson)', 'infl islotes', 'infl septos fibrosos', 'infl acinar', 'necrosisi cel aacin', 'arteritis intima', 'venulitis', 'ductitis', 'infl neural', 'eosinofi', 'neutrofils'

In [4]:
# select the clinical columns we need
# primary label: RECHAZO CLINICO (binary 0/1)
# time context: motivo (1=1wk, 2=1mo, 3=1yr, 4=suspicion, 5=follow-up)
# patient grouping: id paciente (same patient may have multiple studies)
# secondary labels: biopsy-related columns
clinical_cols = [
    "id estudio",
    "id paciente",
    "motivo (1: 1 semana, 2: 1 mes, 3: 1 año, 4: sospecha, 5: seguimiento)=",
    "RECHAZO CLÍNICO",
    "Rechazo confirmadopor biopsia",
    "BIOPSIA",
]

# find biopsy rejection type column (name has inconsistent spacing in CSV)
biopsy_rej_cols = [c for c in clinical_df.columns if "RECHAZO biopsia" in c]
if biopsy_rej_cols:
    clinical_cols.append(biopsy_rej_cols[0])

# check which columns actually exist
available_cols = []
for col in clinical_cols:
    if col in clinical_df.columns:
        available_cols.append(col)
    else:
        print(f"WARNING: column not found: '{col}'")

clinical_subset = clinical_df[available_cols].copy()

# rename for easier use
rename_map = {
    "id estudio": "study_id",
    "id paciente": "patient_id",
    "motivo (1: 1 semana, 2: 1 mes, 3: 1 año, 4: sospecha, 5: seguimiento)=": "motivo",
    "RECHAZO CLÍNICO": "rejection",
    "Rechazo confirmadopor biopsia": "biopsy_confirmed_rejection",
    "BIOPSIA": "biopsy_performed",
}
if biopsy_rej_cols:
    rename_map[biopsy_rej_cols[0]] = "biopsy_rejection_type"

clinical_subset = clinical_subset.rename(columns=rename_map)
print(f"Clinical subset columns: {clinical_subset.columns.tolist()}")

Clinical subset columns: ['study_id', 'patient_id', 'motivo', 'rejection', 'biopsy_confirmed_rejection', 'biopsy_performed', 'biopsy_rejection_type']


In [5]:
# inner join on study_id
merged = pd.merge(radiomics_df, clinical_subset, on="study_id", how="inner")
print(f"Merged: {merged.shape[0]} rows, {merged.shape[1]} columns")

# check for orphans
radiomics_ids = set(radiomics_df["study_id"])
clinical_ids = set(clinical_subset["study_id"])
in_radiomics_not_clinical = radiomics_ids - clinical_ids
in_clinical_not_radiomics = clinical_ids - radiomics_ids

if in_radiomics_not_clinical:
    print(f"In radiomics but not clinical: {in_radiomics_not_clinical}")
if in_clinical_not_radiomics:
    print(f"In clinical but not radiomics: {in_clinical_not_radiomics}")

Merged: 134 rows, 100 columns
In clinical but not radiomics: {'34_02', '47_01', '41_03', '40_02'}


In [6]:
# class distribution
print("Rejection distribution (primary label):")
print(merged["rejection"].value_counts().to_string())
print()

print("Motivo distribution:")
print(merged["motivo"].value_counts().sort_index().to_string())
print()

print("Cross-tab: rejection by motivo")
print(pd.crosstab(merged["motivo"], merged["rejection"], margins=True))

Rejection distribution (primary label):
rejection
0    95
1    39

Motivo distribution:
motivo
1    30
2    36
3    26
4    23
5    19

Cross-tab: rejection by motivo
rejection   0   1  All
motivo                
1          28   2   30
2          28   8   36
3          22   4   26
4           9  14   23
5           8  11   19
All        95  39  134


In [7]:
# save
output_path = os.path.join("reports", "13_merged_radiomics_clinical.csv")
merged.to_csv(output_path, index=False)
print(f"Saved to {output_path}")
print(f"Final shape: {merged.shape}")

merged.head(3)

Saved to reports/13_merged_radiomics_clinical.csv
Final shape: (134, 100)


,study_id,original_firstorder_10Percentile,original_firstorder_90Percentile,original_firstorder_Energy,original_firstorder_Entropy,original_firstorder_InterquartileRange,original_firstorder_Kurtosis,original_firstorder_Maximum,original_firstorder_MeanAbsoluteDeviation,original_firstorder_Mean,...,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength,patient_id,motivo,rejection,biopsy_confirmed_rejection,biopsy_performed,biopsy_rejection_type
0,01_01,26.0,65.0,40347446.0,1.447682,21.0,2.977687,106.0,12.202259,44.459479,...,0.001728,0.947164,0.003742,0.018856,1,1,0,0,0,no
1,01_02,27.0,62.0,24721784.0,1.335817,18.0,3.545282,115.0,10.826002,43.988339,...,0.001628,1.503508,0.004876,0.019147,1,2,1,1,1,1
2,01_03,35.0,72.0,40855206.0,1.425460,20.0,4.058580,124.0,11.664058,52.668379,...,0.001422,1.391369,0.005705,0.014597,1,5,1,1,1,4
